# V14 configuration dictionary

This notebook is a reader-facing reference for adapting
`V14_Thesis_Pipeline.py` to a new binary tabular dataset. It explains
what each configuration family controls, how to choose a valid
experimental design, and how to prepare a project-specific JSON file.

The notebook does not train models or access the study data.

## Goal

Use this notebook to:

- decide whether V14 fits the proposed study;
- select an independent-row, grouped, or temporal split;
- look up configuration fields and allowed values;
- prepare an adapted JSON configuration without changing pipeline code;
- check the configuration structure before a dry run.

V14 supports **binary classification on tabular data**. Multiclass
classification, regression, survival analysis, and unstructured-data
workflows require methodological changes outside this configuration.

## Setup

Run this notebook from the repository root. It reads:

- `docs/V14_CONFIGURATION_DICTIONARY.json`, the field reference;
- `configs/V14_dataset_template.example.json`, the reusable starting point.

Only Python's standard library and IPython display utilities are used.
The original example files are never overwritten.

In [1]:
from copy import deepcopy
from pathlib import Path
import html
import json

from IPython.display import HTML, display

workspace = Path.cwd()
dictionary_path = workspace / "docs" / "V14_CONFIGURATION_DICTIONARY.json"
template_path = workspace / "configs" / "V14_dataset_template.example.json"

dictionary = json.loads(dictionary_path.read_text(encoding="utf-8"))
template = json.loads(template_path.read_text(encoding="utf-8"))
field_count = sum(len(section["fields"]) for section in dictionary["sections"])

print(f"Dictionary version: {dictionary['dictionary_version']}")
print(f"Pipeline version: {dictionary['pipeline_version']}")
print(f"Sections: {len(dictionary['sections'])}")
print(f"Documented fields: {field_count}")

Dictionary version: 1.0
Pipeline version: V14
Sections: 10
Documented fields: 109


## Steps

### 1. Confirm that the study is compatible

Read the compatibility statement before preparing a configuration.
A JSON file can change data, models, budgets, and execution settings,
but it cannot change the statistical scope of the pipeline.

In [2]:
print("COMPATIBILITY")
for name, value in dictionary["compatibility"].items():
    label = name.replace("_", " ").title()
    if isinstance(value, list):
        print(f"- {label}: {', '.join(value)}")
    else:
        print(f"- {label}: {value}")

print("\nDESIGN DECISIONS")
for index, rule in enumerate(dictionary["decision_rules"], start=1):
    print(f"{index}. {rule['question']}")
    print(f"   Yes: {rule['if_yes']}")
    print(f"   No:  {rule['if_no']}")

COMPATIBILITY
- Supported Outcome: Exactly two target classes mapped to 0 and 1.
- Supported Data: Tabular CSV, TSV, Parquet, Pickle, Feather, JSON, pandas DataFrame, or supported scikit-learn demonstration data.
- Row Independence: Use stratified splitting for independent rows and stratified_group splitting for repeated subjects.
- Temporal Data: Use temporal splitting when chronological order defines the test partition.
- Outside Scope: multiclass classification, regression, survival analysis, unstructured image pipelines, text generation

DESIGN DECISIONS
1. Are multiple rows from the same subject or entity present?
   Yes: Set data.group_column and splitting.strategy to stratified_group.
   No:  Set data.group_column to null and use stratified splitting unless time ordering is required.
2. Must the test partition represent earlier or later observations?
   Yes: Set data.timestamp_column and splitting.strategy to temporal.
   No:  Use stratified or stratified_group splitting.
3. Is 

### 2. Choose the split from the structure of the observations

- Use `stratified` when rows are independent and no time ordering is
  required.
- Use `stratified_group` when one subject, patient, household, site, or
  entity can contribute more than one row.
- Use `temporal` when chronology defines the intended validation
  setting.

Group-aware splitting is optional. It should be enabled because the
data structure requires it, not as a default convention.

In [3]:
def recommend_split(case):
    if case.get("time_order_required"):
        return {
            "strategy": "temporal",
            "timestamp_column_required": True,
            "group_column_required": bool(case.get("repeated_entities")),
            "note": "V14 temporal splitting currently requires original-prevalence sampling.",
        }
    if case.get("repeated_entities"):
        return {
            "strategy": "stratified_group",
            "timestamp_column_required": False,
            "group_column_required": True,
            "note": "All rows from one entity remain in one partition.",
        }
    return {
        "strategy": "stratified",
        "timestamp_column_required": False,
        "group_column_required": False,
        "note": "Use only when rows are independent.",
    }

case_profile = {
    "repeated_entities": False,
    "time_order_required": False,
}
split_recommendation = recommend_split(case_profile)
print(json.dumps(split_recommendation, indent=2))

{
  "strategy": "stratified",
  "timestamp_column_required": false,
  "group_column_required": false,
  "note": "Use only when rows are independent."
}


### 3. Browse the field dictionary

Each entry records a JSON path, type, requirement status, default,
allowed values, purpose, and example. Change `selected_section` to
inspect another configuration family.

In [4]:
section_names = [section["name"] for section in dictionary["sections"]]
for index, section in enumerate(dictionary["sections"], start=1):
    print(f"{index:>2}. {section['name']} ({len(section['fields'])} fields)")

def show_section(section_name):
    section = next(
        item for item in dictionary["sections"] if item["name"] == section_name
    )
    rows = []
    for field in section["fields"]:
        allowed = field["allowed"]
        if isinstance(allowed, (list, dict)):
            allowed = json.dumps(allowed, ensure_ascii=False)
        rows.append(
            "<tr>"
            f"<td><code>{html.escape(field['path'])}</code></td>"
            f"<td>{html.escape(str(field['type']))}</td>"
            f"<td>{html.escape(str(field['required']))}</td>"
            f"<td>{html.escape(str(field['default']))}</td>"
            f"<td>{html.escape(str(allowed))}</td>"
            f"<td>{html.escape(field['description'])}</td>"
            f"<td><code>{html.escape(str(field['example']))}</code></td>"
            "</tr>"
        )
    table = (
        f"<h4>{html.escape(section_name)}</h4>"
        "<div style='overflow-x:auto'>"
        "<table>"
        "<thead><tr><th>Path</th><th>Type</th><th>Required</th>"
        "<th>Default</th><th>Allowed</th><th>Meaning</th><th>Example</th>"
        "</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    )
    display(HTML(table))

selected_section = "Data source, outcome, groups, and features"
show_section(selected_section)

 1. Study identity and experiment dimensions (9 fields)
 2. Data source, outcome, groups, and features (16 fields)
 3. Dataset loader dependency (3 fields)
 4. Sampling and splitting (14 fields)
 5. Preprocessing (7 fields)
 6. Models and hyperparameter optimisation (9 fields)
 7. Scenarios and runtime budgets (9 fields)
 8. TabPFN execution, context, and auxiliary comparators (12 fields)
 9. CPU monitoring and energy (13 fields)
10. Outputs, plots, and failure policy (17 fields)


Path,Type,Required,Default,Allowed,Meaning,Example
data.source,string path,"One of source, dataframe, or sklearn_dataset",None,"CSV, TSV, Parquet, Pickle, Feather, or JSON file",Location of the tabular dataset. Keep confidential data outside Git.,data/analysis.parquet
data.dataframe,pandas DataFrame,Python API only; alternative to data.source,None,In-memory DataFrame,Programmatic input for module use. It cannot be represented directly in JSON.,analysis_frame
data.sklearn_dataset,string,No,None,"[""breast_cancer"", ""iris_binary"", ""wine_binary""]",Small public dataset used for smoke tests rather than thesis evidence.,breast_cancer
data.dataset_identifier,string,Recommended,None,Versioned non-sensitive identifier,Traceable dataset label stored in manifests without copying raw data.,cohort_release_2026_01
data.read_kwargs,object,No,{},Keyword arguments accepted by the selected pandas reader,"Reader-specific options such as delimiters, encodings, or column types.","{'sep': ';', 'encoding': 'utf-8'}"
data.target,string,Yes unless target_fn is used through the Python API,None,Existing outcome column,Binary outcome column. Rows with missing outcomes are removed.,outcome
data.positive_label,scalar,Recommended when the positive class is not already 1,Automatic detection,One of the two observed outcome values,Outcome value mapped to encoded class 1.,case
data.target_map,object,No,None,Complete mapping of both target values to 0 and 1,Explicit binary encoding that overrides positive_label.,"{'control': 0, 'case': 1}"
data.group_column,string or null,Required for grouped splitting,None,Existing non-missing identifier column,Subject/entity identifier that prevents the same group appearing in training and test data.,patient_id
data.timestamp_column,string or null,Required for temporal splitting,None,Existing chronologically sortable column,Time variable used to construct temporal partitions.,observation_time


### 4. Adapt the reusable template

The example below creates an in-memory configuration for an independent-row
dataset. Replace every illustrative name with values from the study's
data dictionary. `positive_label` must identify the class that will be
mapped to 1.

Keep `enabled` false and `template_only` true while the file is still a
draft. Change both flags only after the validation and dry-run steps.

In [5]:
adapted = deepcopy(template)

adapted["experiment_name"] = "my_binary_classification_study"
adapted["iterations"] = 5
adapted["base_seed"] = 2025
adapted["sample_sizes"] = [100, "full"]

adapted["data"].update(
    {
        "source": "data/my_dataset.csv",
        "dataset_identifier": "my_dataset_v1",
        "target": "outcome",
        "positive_label": 1,
        "group_column": None,
        "timestamp_column": None,
        "feature_columns": None,
        "drop_columns": ["record_id"],
        "datetime_columns": [],
    }
)

adapted["sampling"]["strategy"] = "original_prevalence"
adapted["splitting"].update(
    {
        "strategy": "stratified",
        "group_aware": False,
        "require_groups": False,
    }
)
adapted["outputs"]["root"] = "experiment_outputs/my_study"

preview = {
    "experiment_name": adapted["experiment_name"],
    "iterations": adapted["iterations"],
    "sample_sizes": adapted["sample_sizes"],
    "data": adapted["data"],
    "sampling": adapted["sampling"],
    "splitting": adapted["splitting"],
    "outputs": adapted["outputs"],
}
print(json.dumps(preview, indent=2))

{
  "experiment_name": "my_binary_classification_study",
  "iterations": 5,
  "sample_sizes": [
    100,
    "full"
  ],
  "data": {
    "source": "data/my_dataset.csv",
    "dataset_identifier": "my_dataset_v1",
    "read_kwargs": {},
    "target": "outcome",
    "positive_label": 1,
    "group_column": null,
    "timestamp_column": null,
    "feature_columns": null,
    "drop_columns": [
      "record_id"
    ],
    "datetime_columns": [],
    "drop_constant": true,
    "max_cardinality": null,
    "keep_dataframe": true
  },
  "sampling": {
    "strategy": "original_prevalence"
  },
  "splitting": {
    "strategy": "stratified",
    "group_aware": false,
    "require_groups": false,
    "strict": true,
    "candidate_splits": 256,
    "min_class_count_per_partition": 2,
    "shuffle_rows_within_partitions": true,
    "temporal_window": "latest",
    "temporal_gap_rows": 0,
    "temporal_enforce_group_disjoint": false
  },
  "outputs": {
    "root": "experiment_outputs/my_study",
   

### 5. Apply the matching data-design recipe

**Independent rows**

```json
"data": {"group_column": null, "timestamp_column": null},
"splitting": {
  "strategy": "stratified",
  "group_aware": false,
  "require_groups": false
}
```

**Repeated subjects or entities**

```json
"data": {"group_column": "subject_id", "timestamp_column": null},
"splitting": {
  "strategy": "stratified_group",
  "group_aware": true,
  "require_groups": true
}
```

**Chronological validation**

```json
"data": {"group_column": null, "timestamp_column": "observation_time"},
"sampling": {"strategy": "original_prevalence"},
"splitting": {"strategy": "temporal", "temporal_window": "latest"}
```

For temporal data with repeated entities, review
`temporal_enforce_group_disjoint` and the study design before execution.

### 6. Define models, tuning, and runtime budgets

A model is available only when its configuration is enabled and its
package is installed. `requirements-core.txt` covers the core scientific
stack; `requirements-optional.txt` lists optional learners and reporting
tools.

Budget scenarios use one declared reference model:

- for a tuned reference model, the budget basis is its Optuna tuning-loop
  runtime;
- for a non-tuned reference model, the basis is its configured reference
  execution runtime;
- for an unbudgeted comparison, set the budget type to `none`.

The same declared basis must be used when interpreting timing results.

In [6]:
enabled_models = [
    name for name, settings in adapted["models"].items()
    if settings.get("enabled", False)
]

scenario_summary = []
for scenario_name, scenario in adapted["scenarios"].items():
    scenario_summary.append(
        {
            "name": scenario_name,
            "budget_type": (
                "budgeted" if scenario["budgeting"]["enabled"] else "none"
            ),
            "budget_reference_model": scenario.get("budget_reference_model"),
            "models": scenario["enabled_models"],
        }
    )

print("Enabled model configurations:", ", ".join(enabled_models))
print(json.dumps(scenario_summary, indent=2))

Enabled model configurations: TabPFN, L-SLR, Augmented_SLR, RandomForest, XGBoost, CatBoost
[
  {
    "name": "Replace_With_Scenario_Name",
    "budget_type": "none",
    "budget_reference_model": null,
    "models": [
      "L-SLR",
      "Augmented_SLR",
      "RandomForest",
      "XGBoost",
      "CatBoost",
      "TabPFN"
    ]
  }
]


### 7. Set TabPFN context, CPU use, energy logging, and outputs

For local TabPFN execution, `local_device: "auto"` selects an available
device without requiring one. Context can be fixed, adaptive, or full.
Adaptive context is generally more portable because it scales from the
available training rows while respecting configured limits.

CPU monitoring reports available logical processors and utilization
during model runs. It does not reserve hardware. Use `max_available` for
the detected limit or `explicit` with a reviewed thread count.

Energy tracking is optional. If enabled, verify the CodeCarbon setup and
document the measurement environment. Output directories may contain
predictions, row identifiers, or derived sensitive information and
should be handled under the study's data-governance rules.

Fitted-model serialization remains disabled in the thesis pipeline.

In [7]:
operational_view = {
    "tabpfn_execution": adapted["models"]["TabPFN"]["execution"],
    "tabpfn_context": adapted["models"]["TabPFN"]["local_tabpfn_budget"][
        "context_strategy"
    ],
    "cpu_parallelism": adapted["cpu_parallelism"],
    "cpu_monitoring": adapted["cpu_monitoring"],
    "energy": adapted["energy"],
    "outputs": adapted["outputs"],
    "failure_policy": {
        "on_model_error": adapted["execution"]["on_model_error"],
        "on_iteration_error": adapted["execution"]["on_iteration_error"],
    },
}
print(json.dumps(operational_view, indent=2))

{
  "tabpfn_execution": {
    "path": "local",
    "local_device": "auto",
    "require_requested_device": false
  },
  "tabpfn_context": {
    "strategy": "adaptive",
    "fraction": 0.2,
    "fraction_denominator": "outer_train"
  },
  "cpu_parallelism": {
    "policy": "max_available",
    "threads": "auto",
    "prevent_nested_oversubscription": true,
    "allow_oversubscription": false,
    "environment": {}
  },
  "cpu_monitoring": {
    "enabled": true,
    "show_console": true,
    "sampling_interval_seconds": 5,
    "save_timeseries": true
  },
  "energy": {
    "enabled": false,
    "required": false,
    "allow_unavailable": true,
    "tracking_mode": "process",
    "save_client_side_cloud_energy": true,
    "save_codecarbon_metadata": true
  },
  "outputs": {
    "root": "experiment_outputs/my_study",
    "save_fitted_models": false,
    "save_predictions_parquet": true,
    "save_predictions_npz": true,
    "save_group_ids": true,
    "artifact_index": true
  },
  "failure

### 8. Save and run only after review

When the adapted configuration is ready:

1. remove or resolve all `replace_with_...` placeholders;
2. set `template_only` to false;
3. set `enabled` to true if it should be discovered through
   `--config-dir`;
4. save it under a new name such as `configs/my_study.json`;
5. run the pipeline's self-test and dry run before a full experiment.

Example commands:

```powershell
python V14_Thesis_Pipeline.py --self-test
python V14_Thesis_Pipeline.py --config configs/my_study.json --dry-run
python V14_Thesis_Pipeline.py --config configs/my_study.json
```

In [8]:
proposed_path = workspace / "configs" / "my_study.json"
proposed_text = json.dumps(adapted, indent=2, ensure_ascii=False) + "\n"

print(f"Proposed destination: {proposed_path}")
print(f"Prepared JSON characters: {len(proposed_text):,}")
print("No file was written by this notebook.")

Proposed destination: C:\Users\Sina\Downloads\Thesis Analysis\Thesis Code - dynamic pipeline\configs\my_study.json
Prepared JSON characters: 4,194
No file was written by this notebook.


## Checks

These checks verify the documentation assets and the structural choices
made in the example configuration. They do not inspect the dataset,
evaluate leakage, or confirm that optional model packages are installed.
The V14 dry run performs the next level of validation.

In [9]:
checks = []

def record(name, condition):
    checks.append((name, bool(condition)))

documented_paths = [
    field["path"]
    for section in dictionary["sections"]
    for field in section["fields"]
]
required_top_level = {
    "extends", "experiment_name", "data", "sampling", "splitting",
    "preprocessing", "models", "scenarios", "cpu_parallelism",
    "cpu_monitoring", "energy", "outputs", "plots", "execution",
}

record("dictionary identifies V14", dictionary["pipeline_version"] == "V14")
record("dictionary paths are unique", len(documented_paths) == len(set(documented_paths)))
record("template contains required top-level keys", required_top_level <= set(template))
record("adapted configuration is valid JSON", json.loads(proposed_text) == adapted)
record(
    "split strategy is supported",
    adapted["splitting"]["strategy"]
    in {"auto", "stratified", "stratified_group", "temporal"},
)
record(
    "group column is present when group splitting is selected",
    adapted["splitting"]["strategy"] != "stratified_group"
    or adapted["data"]["group_column"] is not None,
)
record(
    "timestamp is present when temporal splitting is selected",
    adapted["splitting"]["strategy"] != "temporal"
    or adapted["data"]["timestamp_column"] is not None,
)
record(
    "temporal splitting uses original prevalence",
    adapted["splitting"]["strategy"] != "temporal"
    or adapted["sampling"]["strategy"] == "original_prevalence",
)
record(
    "fitted-model serialization remains disabled",
    adapted["outputs"]["save_fitted_models"] is False,
)

for name, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

if not all(passed for _, passed in checks):
    raise AssertionError("One or more configuration checks failed.")

print(f"\nCONFIGURATION DICTIONARY CHECK: PASS ({len(checks)}/{len(checks)})")

PASS - dictionary identifies V14
PASS - dictionary paths are unique
PASS - template contains required top-level keys
PASS - adapted configuration is valid JSON
PASS - split strategy is supported
PASS - group column is present when group splitting is selected
PASS - timestamp is present when temporal splitting is selected
PASS - temporal splitting uses original prevalence
PASS - fitted-model serialization remains disabled

CONFIGURATION DICTIONARY CHECK: PASS (9/9)


## Next Steps

1. Copy `configs/V14_dataset_template.example.json` to a new,
   study-specific JSON file.
2. Complete `docs/DATA_DICTIONARY_TEMPLATE.md` before mapping columns
   into the configuration.
3. Confirm the unit of observation, outcome definition, positive class,
   leakage exclusions, grouping structure, and time-order requirements
   with the study protocol.
4. Install the core requirements and only the optional packages required
   by the selected models.
5. Run `--self-test`, then `--dry-run`, then a small quality-control
   scenario before the full experiment.
6. Archive the final configuration, environment information, input
   fingerprint, seeds, logs, and outputs with the study record.
7. Before publishing the repository, complete the license and citation
   details in `docs/RELEASE_CHECKLIST.md`.

The companion notebook
`V14_Thesis_Pipeline_Reader_Guide.ipynb` explains the complete execution
flow and output interpretation.